### mmr technique

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models.base import init_chat_model
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS


d:\RAG System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader=TextLoader("langchain_sample.txt")
docs=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)
chunks=text_splitter.split_documents(docs)

embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1838.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm=init_chat_model("groq:openai/gpt-oss-safeguard-20b")
llm

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x000001789E7EA270>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001789E7EB770>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
retriever=vectorstore.as_retriever(search_kwargs={"k":5},
                                   search_type="mmr",)

In [11]:
prompt=PromptTemplate.from_template("""
answer the question based on the following retrieved contexts:
contexts:{context}
question:{input}                                       
                                                                               
""")

In [12]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [14]:
chain_docs=create_stuff_documents_chain(llm=llm,prompt=prompt)
chain=create_retrieval_chain(retriever= retriever,combine_docs_chain=chain_docs)

In [15]:
query="What is Langchain?"
response=chain.invoke({"input":query,"context":chain_docs})
print(response["answer"])

**LangChain** is a flexible, open‑source framework for building applications that use large language models (LLMs). It offers a set of tools and abstractions—such as prompt templates, chains, memory, and agents—to make LLMs more effective and manageable. With LangChain, developers can:

- **Integrate** with many third‑party LLM providers (OpenAI, Hugging Face, Cohere, etc.).
- **Manage prompts** and chain together multiple LLM calls.
- **Add memory** to keep context across multi‑step conversations or tasks.
- **Build agents** that decide which tools (e.g., web search, calculators, APIs) to call for complex problems.
- **Implement Retrieval‑Augmented Generation (RAG)** using vector databases like FAISS, Chroma, or Pinecone, and combine sparse (BM25) and dense retrieval for better document search.

In short, LangChain is a comprehensive toolkit that simplifies the creation of stateful, data‑aware, and multi‑step LLM applications.
